# Exploração e KPIs — Criptoativos declarados à Receita Federal

Notebook de apoio ao Projeto Integrador Extensionista (Técnicas de Visualização de Dados e Business Intelligence).
Ele percorre o mesmo caminho do pipeline (`src/`), passo a passo, para documentar a análise.

**Fonte:** planilha de dados abertos da Receita Federal (IN RFB nº 1.888/2019), versão 26/08/2026, em `data/raw/`.

**Organização parceira / decisão apoiada:** preencher em `src/config.py`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # raiz do projeto

import pandas as pd
import plotly.io as pio
from src import extract, clean, quality, kpis, charts, insights, config

pd.options.display.float_format = "{:,.2f}".format
pio.renderers.default = "notebook_connected"  # precisa de internet só para visualizar os gráficos

## 1. Extração
A planilha tem 4 relatórios (abas) com texto explicativo no topo e meses por extenso. `extract.py` localiza as linhas de dados.

In [2]:
brutos = extract.extrair_tudo()
for nome, df in brutos.items():
    print(nome, df.shape)
brutos["r1"].head()

r1 (83, 9)
r2 (83, 3)
r3 (83, 5)
r4 (4519, 5)


,mes,ext_pf,ext_pj,ext_subtotal,sem_pf,sem_pj,sem_subtotal,exchange_br,total_geral
0,Agosto de 2019,376.25,745.33,"1,121.58",114.88,269.54,384.42,"2,531.02","4,037.01"
1,Setembro de 2019,249.55,"1,955.45","2,205.00",191.31,721.82,913.13,"2,757.33","5,875.46"
2,Outubro de 2019,305.66,"1,238.60","1,544.26",314.42,"1,178.71","1,493.13","3,458.50","6,495.89"
3,Novembro de 2019,16.58,"1,529.78","1,546.36",1.24,"1,007.97","1,009.21","2,974.60","5,530.18"
4,Dezembro de 2019,86.08,"1,139.89","1,225.97",80.99,"2,377.19","2,458.17","3,056.43","6,740.57"


## 2. Limpeza
Datas viram `datetime` (1º dia do mês), números viram `float`/`int`, e criamos colunas derivadas (participações, ticket relativo, grupo do ativo).

In [3]:
d = clean.limpar_tudo(brutos)
d["r1"].dtypes

mes                         datetime64[us]
ext_pf                             float64
ext_pj                             float64
ext_subtotal                       float64
sem_pf                             float64
sem_pj                             float64
sem_subtotal                       float64
exchange_br                        float64
total_geral                        float64
pf_declarante_rs_mi                float64
pj_declarante_rs_mi                float64
base_autodeclarada_rs_mi           float64
share_pf_autodeclarada             float64
share_exchange_br                  float64
share_exchange_ext                 float64
share_sem_exchange                 float64
total_geral_bi                     float64
dtype: object

In [4]:
d["r4"].head()

,criptoativo,mes,n_operacoes,valor_total,valor_medio,grupo,valor_total_mi
0,AAVE,2020-08-01,1,"59,638.94","59,638.94",Outros ativos,0.06
1,AAVE,2021-01-01,32,"296,891.77","9,277.87",Outros ativos,0.30
2,AAVE,2021-02-01,129,"974,326.69","7,552.92",Outros ativos,0.97
3,AAVE,2021-03-01,103,"64,072.39",622.06,Outros ativos,0.06
4,AAVE,2021-04-01,2859,"2,948,464.51","1,031.29",Outros ativos,2.95


## 3. Qualidade dos dados
Antes de interpretar, verificamos se a planilha é coerente por dentro e entre abas.

In [5]:
quality.validar(d)

,verificacao,status,detalhe
0,R1: PF + PJ = subtotal (exterior),OK,erro máx. 0.000000 R$ mi
1,R1: PF + PJ = subtotal (sem exchange),OK,erro máx. 0.000000 R$ mi
2,R1: subtotais + exchanges BR = total geral,OK,erro máx. 0.000000 R$ mi
3,R3: % feminino + masculino = 100 (operações),OK,desvio máx. 0.000 p.p.
4,R3: % feminino + masculino = 100 (valor),OK,desvio máx. 0.000 p.p.
5,R4: valor médio = valor total / nº de operações,OK,erro máx. R$ 0.000000
6,R4: sem duplicatas (ativo × mês),OK,0 duplicatas
7,R4: sem valores negativos,OK,0 ocorrências
8,R4: sem valores nulos,OK,0 células vazias
9,Séries mensais contínuas (sem meses faltando),OK,"08/2019 a 06/2026, 83 meses"


In [6]:
quality.detectar_quebras(d["r2"])

,mes,serie,valor,valor_mes_anterior,razao_sobre_mes_anterior,tipo
0,2022-04-01,CNPJ,35084,14109,2.49,salto
1,2023-08-01,CPF,9078782,4406776,2.06,salto
2,2023-12-01,CPF,1951011,9208489,0.21,queda
3,2024-01-01,CPF,8888313,1951011,4.56,salto
4,2024-03-01,CNPJ,23874,421419,0.06,queda
5,2024-10-01,CPF,1884039,4567521,0.41,queda
6,2024-12-01,CNPJ,84816,26024,3.26,salto
7,2024-12-01,CPF,6190941,2999797,2.06,salto
8,2025-06-01,CNPJ,25578,110683,0.23,queda
9,2025-06-01,CPF,1328657,5266356,0.25,queda


## 4. KPIs
### 4.1 Volume declarado

In [7]:
vol = kpis.kpi_volume(d["r1"])
vol.tail(3)

,mes,volume_bi,acum_12m_bi,yoy_mensal,yoy_acum_12m
80,2026-04-01,43.52,519.34,0.18,0.17
81,2026-05-01,58.07,541.30,0.61,0.20
82,2026-06-01,57.54,559.78,0.47,0.22


In [8]:
charts.fig_volume(vol)

### 4.2 Canais de intermediação

In [9]:
charts.fig_canais(kpis.kpi_canais(d["r1"]))

### 4.3 Composição por ativo — stablecoins, Bitcoin, Ethereum

In [10]:
charts.fig_composicao(kpis.kpi_composicao(d["r4"]))

In [11]:
kpis.kpi_ativos(d["r4"], inicio=d["r4"]["mes"].max() - pd.DateOffset(months=11)).head(10)

,criptoativo,grupo,valor_total,n_operacoes,ticket_medio,participacao
0,USDT,Stablecoins,"374,462,263,022.22",20144546,"18,588.77",0.74
1,USDC,Stablecoins,"62,193,332,837.52",31061274,"2,002.28",0.12
2,BTC,Bitcoin,"37,015,632,333.53",28366289,"1,304.92",0.07
3,ETH,Ethereum,"12,954,375,473.33",10016557,"1,293.30",0.03
4,SOL,Outros ativos,"5,384,402,261.46",5410225,995.23,0.01
5,XRP,Outros ativos,"3,201,040,865.52",4059190,788.59,0.01
6,BRZ,Stablecoins,"2,807,733,881.62",4316857,650.41,0.01
7,UNI,Outros ativos,"1,196,528,685.44",1586555,754.17,0.00
8,LINK,Outros ativos,"910,277,088.51",1432094,635.63,0.00
9,AAVE,Outros ativos,"804,231,155.20",1026832,783.22,0.00


### 4.4 Declarantes únicos (com quebras sinalizadas)

In [12]:
charts.fig_declarantes(kpis.kpi_declarantes(d["r2"]), quality.detectar_quebras(d["r2"]))

### 4.5 Gênero

In [13]:
charts.fig_genero(kpis.kpi_genero(d["r3"]))

## 5. Achados (Data Storytelling)
Textos calculados a partir dos números reais.

In [14]:
for a in insights.gerar_achados(d):
    print(f"• {a['titulo']}\n  {a['texto']}\n")

• O volume declarado continua em expansão
  Nos 12 meses até 06/2026, foram declarados R$ 559,8 bilhões em operações com criptoativos, 21,6% acima dos 12 meses anteriores. O maior mês da série foi 05/2026, com R$ 58,1 bilhões.

• As stablecoins viraram o centro do mercado
  Nos últimos 12 meses, stablecoins somam 87,0% do valor negociado nos principais ativos (eram 10,6% nos 12 primeiros meses da série), enquanto o Bitcoin caiu de 57,3% para 7,3%. O USDT sozinho responde por 74,1% do valor.

• Mercado altamente concentrado em poucos ativos
  Em 06/2026, os 5 maiores ativos concentram 98,9% do valor (HHI de 5.061 em 10.000). Acompanhar a diversidade de ativos é tão importante quanto acompanhar o volume.

• Exchanges no Brasil seguem à frente, mas o peso dos canais mudou
  Nos últimos 12 meses, exchanges no Brasil intermediaram 62,4% do volume e perderam participação frente aos 12 meses anteriores (69,7%). As operações sem exchange (P2P) passaram de 17,0% para 22,4%.

• Mulheres particip

## 6. Próximos passos
1. Preencher a organização parceira em `src/config.py` e registrar as interações em `docs/registro_interacoes.md`.
2. Validar os KPIs com o representante da organização.
3. Ajustar o painel (`app/dashboard.py`) às perguntas reais dela.